In [ ]:
import pandas as pd
import numpy as np
import math
from pathlib import Path

import equiboots as eqb
import matplotlib.pyplot as plt
from equiboots.tables import metrics_table
from core.model_registry import best_per_algo, load_best_per_algo

In [ ]:
help(eqb)

## Read in Data and Model Object

In [ ]:
X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet").squeeze()

In [ ]:
best_per_algo(metric="valid Average Precision")
champs = load_best_per_algo(metric="valid Average Precision")
model_catboost = champs["cat_outcome"]
model_catboost_no_sex = champs["cat_outcome_no_sex"]

In [ ]:
X_valid, y_valid = model_catboost_no_sex.get_valid_data(X, y)
X_test, y_test = model_catboost_no_sex.get_test_data(X, y)
y_test = y_test["outcome"]

In [ ]:
X_test_labeled = X_test.copy()
X_test_labeled["sex"] = X_test_labeled["sex"].map({1: "Male", 0: "Female"}).astype(str)

fairness_df = X_test_labeled[["sex"]].reset_index()

In [ ]:
X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet")["outcome"].squeeze()
X_test, y_test = model_catboost.get_test_data(X, y)

## Bootstrap Estimates

Bootstrap estimates:m
- randomly sampling fairness_df, y_true, y_prob, and y_pred

In [ ]:
test_config = {
    "test_type": "bootstrap_test",
    "alpha": 0.05,
    "adjust_method": "bonferroni",
    "confidence_level": 0.95,
    "tail_type": "two_tailed",
    "metrics": [
        "Accuracy_diff",
        "Precision_diff",
        "Recall_diff",
        "F1_Score_diff",
        "Specificity_diff",
        "TP_Rate_diff",
        "FP_Rate_diff",
        "FN_Rate_diff",
        "TN_Rate_diff",
        "Prevalence_diff",
        "Predicted_Prevalence_diff",
        "ROC_AUC_diff",
        "Average_Precision_Score_diff",
        "Log_Loss_diff",
        "Brier_Score_diff",
        "Calibration_AUC_diff",
    ],
}

In [ ]:
y_test_arr = np.asarray(y_test)          # non-destructive; safe to re-run
int_list = np.linspace(0, len(y_test_arr), num=len(y_test_arr), dtype=int).tolist()


def run_audit(model, label):
    """Bootstrap fairness audit for one model. Returns (metrics, sig_tests)."""
    thr = model.threshold["average_precision"]
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= thr).astype(int)

    eq = eqb.EquiBoots(
        y_true=y_test_arr,
        y_pred=y_pred,
        y_prob=y_prob,
        fairness_df=fairness_df,
        fairness_vars=["sex"],
        seeds=int_list,
        reference_groups=["Male"],
        task="binary_classification",
        bootstrap_flag=True,
        num_bootstraps=5001,
        boot_sample_size=len(y_test_arr),
        group_min_size=1,
        balanced=False,
        stratify_by_outcome=True,
    )
    eq.set_fix_seeds(int_list)
    eq.grouper(groupings_vars=["sex"])

    boot_data = eq.slicer("sex")
    boot_metrics = eq.get_metrics(boot_data)
    diffs = eq.calculate_differences(boot_metrics, "sex")
    sig = eq.analyze_statistical_significance(
        metric_dict=boot_metrics,
        var_name="sex",
        test_config=test_config,
        differences=diffs,
    )
    print(f"{label}: threshold {thr:.3f}")
    return boot_metrics, diffs, sig

In [ ]:
boots_primary, diffs_primary, sig_primary = run_audit(model_catboost, "primary")
boots_ablated, diffs_ablated, sig_ablated = run_audit(model_catboost_no_sex, "ablated")

In [ ]:
def relabel(boots, prefix):
    """Prefix group names so two models can share one plot."""
    return [{f"{prefix}: {g}": m for g, m in rep.items()} for rep in boots]

combined = [
    {**a, **b}
    for a, b in zip(
        relabel(boots_primary, "1 With sex"),
        relabel(boots_ablated, "2 No sex"),
    )
]

In [ ]:
WITH_SEX = "#4C72B0"
NO_SEX = "#C44E52"
REF_GREY = "#888888"

# eq_plot_bootstrap_forest calls plt.show() internally, which detaches the
# figure under the inline backend. Intercept it so the figure survives and
# can be recolored.
captured = {}
real_show = plt.show
plt.show = lambda *a, **k: captured.setdefault("fig", plt.gcf())

try:
    eqb.eq_plot_bootstrap_forest(
        group_boot_metrics=combined,
        metric="ROC AUC",
        reference_group="1 With sex: Male",
        title="AUROC by sex, before and after removing sex",
        figsize=(8, 4.5),
        x_lim=(0.55, 1.02),
        sort_alphabetically=True,
    )
finally:
    plt.show = real_show

fig = captured["fig"]
ax = fig.axes[0]

labels = [t.get_text() for t in ax.get_yticklabels()]
print(labels)  # verify row order before trusting the color map

row_color = [WITH_SEX if "With sex" in lab else NO_SEX for lab in labels]

# error bars: three black line objects per row (span, left cap, right cap)
bars = [ln for ln in ax.get_lines() if ln.get_color() in ("k", "black")]
for i, ln in enumerate(bars):
    ln.set_color(row_color[i // 3])

# mean markers
for coll in ax.collections:
    coll.set_color(row_color)

# strip the numeric sort prefixes: "1 With sex: Male" -> "With sex: Male"
ax.set_yticklabels([lab.split(" ", 1)[1] for lab in labels])
ax.set_xlabel("AUROC (95% CI)")
ax.set_ylabel("Model and sex")

# recolor the reference line so it is not mistaken for a group series
for ln in ax.get_lines():
    if ln.get_linestyle() == "--":
        ln.set_color(REF_GREY)

# the legend was built before the recolor, so update its handle to match
leg = ax.get_legend()
if leg is not None:
    handles = getattr(leg, "legend_handles", None) or leg.legendHandles
    for handle in handles:
        if handle.get_linestyle() == "--":
            handle.set_color(REF_GREY)

fig.savefig("../images/pdf_images/forest_auroc_combined.pdf",
            bbox_inches="tight")
fig.savefig("../images/svg_images/fig_5_forest_auroc_combined.svg",
            bbox_inches="tight")
fig.savefig("../images/tiff_images/fig_5_forest_auroc_combined.tiff",
            bbox_inches="tight", dpi=600, pil_kwargs={"compression": "tiff_lzw"},)

In [ ]:
"""
Decomposition of the sex-based false positive rate disparity.

The observed FPR ratio factors exactly into three multiplicative components
(Chouldechova identity). Because the factors multiply, their shares are
additive on the log scale, so the stacked bars are drawn in log space. Drawing
the raw factors as a stacked bar would imply they add, which they do not.

Every number is computed from the confusion matrices rather than transcribed,
and the identity is asserted before anything is plotted.
"""

# ---------------------------------------------------------------------------
# Test-set confusion matrices, primary model, threshold 0.080
# ---------------------------------------------------------------------------
MEN = dict(tp=10, fp=28, fn=6, tn=85)
WOMEN = dict(tp=4, fp=7, fn=3, tn=95)

FIG_NAME = "fig_4_fpr_decomposition"
OUT_PNG = f"../images/png_images/{FIG_NAME}.png"
OUT_PDF = f"../images/pdf_images/{FIG_NAME}.pdf"
OUT_SVG = f"../images/svg_images/{FIG_NAME}.svg"
OUT_TIFF = f"../images/tiff_images/{FIG_NAME}.tiff"

TIFF_KW = {"dpi": 600, "bbox_inches": "tight",
           "pil_kwargs": {"compression": "tiff_lzw"}}


def rates(cm):
    """Group rates, with the Chouldechova identity checked against observed FPR."""
    tp, fp, fn, tn = cm["tp"], cm["fp"], cm["fn"], cm["tn"]
    n = tp + fp + fn + tn
    p = (tp + fn) / n
    ppv = tp / (tp + fp)
    fnr = fn / (tp + fn)
    fpr = fp / (fp + tn)
    identity = (p / (1 - p)) * ((1 - ppv) / ppv) * (1 - fnr)
    assert abs(fpr - identity) < 1e-12, "identity does not reproduce FPR"
    return dict(n=n, p=p, ppv=ppv, fnr=fnr, fpr=fpr)


m, w = rates(MEN), rates(WOMEN)

f_prev = (m["p"] / (1 - m["p"])) / (w["p"] / (1 - w["p"]))
f_ppv = ((1 - m["ppv"]) / m["ppv"]) / ((1 - w["ppv"]) / w["ppv"])
f_fnr = (1 - m["fnr"]) / (1 - w["fnr"])
ratio = m["fpr"] / w["fpr"]

assert abs(f_prev * f_ppv * f_fnr - ratio) < 1e-9, "factors do not reproduce ratio"

logs = [math.log(f_prev), math.log(f_ppv), math.log(f_fnr)]
total_log = math.log(ratio)
shares = [x / total_log for x in logs]

print(f"FPR men   = {m['fpr']:.4f}   FPR women = {w['fpr']:.4f}")
print(f"ratio     = {ratio:.4f}")
print(f"  prevalence odds {f_prev:.4f}  share {shares[0]*100:.1f}%")
print(f"  PPV             {f_ppv:.4f}  share {shares[1]*100:.1f}%")
print(f"  FNR             {f_fnr:.4f}  share {shares[2]*100:.1f}%")

# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------
COLORS = ["#4C72B0", "#DD8452", "#937860"]
LABELS = ["Prevalence odds", "PPV", "FNR"]

fig, (axA, axB) = plt.subplots(
    2, 1, figsize=(7.6, 5.0), gridspec_kw={"height_ratios": [1.15, 1]}
)

# Panel A: multiplicative chain on a log axis, so segment widths are log(factor)
left = 0.0
for lg, color, label, factor in zip(logs, COLORS, LABELS, [f_prev, f_ppv, f_fnr]):
    axA.barh(0, lg, left=left, height=0.5, color=color, edgecolor="white",
             linewidth=1.2, label=f"{label} ({factor:.2f}x)")
    if lg > 0.06:
        axA.text(left + lg / 2, 0, f"{factor:.2f}x", ha="center", va="center",
                 color="white", fontsize=10, fontweight="bold")
    left += lg

axA.set_xlim(-0.02, total_log * 1.04)
axA.set_ylim(-0.55, 0.75)
axA.set_yticks([])
# last tick lands exactly on the computed ratio instead of a hardcoded value
tick_factors = [f for f in (1.0, 1.5, 2.0, 2.5, 3.0) if f < ratio] + [ratio]
axA.set_xticks([math.log(f) for f in tick_factors])
axA.set_xticklabels([f"{f:.2f}x" for f in tick_factors], fontsize=9)
axA.set_xlabel("Cumulative multiplicative factor (log scale)", fontsize=9.5)
axA.set_title(
    f"Observed false positive rate ratio, men to women: "
    f"{m['fpr']:.3f} / {w['fpr']:.3f} = {ratio:.2f}x",
    fontsize=10.5, fontweight="bold", pad=10,
)
for side in ("top", "right", "left"):
    axA.spines[side].set_visible(False)
axA.legend(loc="upper center", bbox_to_anchor=(0.5, -0.42), ncol=3,
           fontsize=8.5, frameon=False)

# Panel B: share of the disparity, additive on the log scale
left = 0.0
for share, color in zip(shares, COLORS):
    axB.barh(0, share * 100, left=left, height=0.5, color=color,
             edgecolor="white", linewidth=1.2)
    if share > 0.05:
        axB.text(left + share * 100 / 2, 0, f"{share*100:.1f}%", ha="center",
                 va="center", color="white", fontsize=10, fontweight="bold")
    left += share * 100

axB.set_xlim(0, 100)
axB.set_ylim(-0.45, 0.78)
axB.set_yticks([])
axB.set_xticks([0, 20, 40, 60, 80, 100])
axB.set_xticklabels(["0%", "20%", "40%", "60%", "80%", "100%"], fontsize=9)
axB.set_xlabel("Share of the disparity (log scale)", fontsize=9.5)
axB.set_title("Attribution of the disparity", fontsize=10.5,
              fontweight="bold", pad=26)
for side in ("top", "right", "left"):
    axB.spines[side].set_visible(False)

axB.axvline(shares[0] * 100, color="#B00000", linewidth=1.4, linestyle="--",
            ymin=0.02, ymax=0.72)
axB.text(shares[0] * 100 / 2, 0.40,
         f"attributable to base rate ({shares[0]*100:.1f}%)",
         ha="center", va="bottom", fontsize=8.5, color="#333333")
axB.text(shares[0] * 100 + (100 - shares[0] * 100) / 2, 0.40,
         f"addressable ({(1-shares[0])*100:.1f}%)",
         ha="center", va="bottom", fontsize=8.5, color="#333333")

plt.tight_layout()
plt.subplots_adjust(hspace=1.35)

for path in (OUT_PNG, OUT_PDF, OUT_SVG, OUT_TIFF):
    Path(path).parent.mkdir(parents=True, exist_ok=True)

fig.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
fig.savefig(OUT_PDF, bbox_inches="tight")
fig.savefig(OUT_SVG, bbox_inches="tight")
fig.savefig(OUT_TIFF, **TIFF_KW)
plt.show()

In [ ]:
import re
from pathlib import Path

wanted_metrics = [
    "ROC_AUC_diff",
    "FP_Rate_diff",
    "Predicted_Prevalence_diff",
    "Recall_diff",
]

title_map = {
    "roc auc": "AUROC difference",
    "fp rate": "FPR difference",
    "predicted prevalence": "Predicted prevalence difference",
    "recall": "Sensitivity difference",
}

# Group names as they may appear in the diffs (sex coded 1 = male)
group_labels = {
    "female": "Women",
    "male": "Men",
    "0": "Women",
    "1": "Men",
}


def to_people(text):
    """Replace female/male with women/men in any rendered label."""
    text = re.sub(r"\bfemales?\b", "Women", text, flags=re.IGNORECASE)
    text = re.sub(r"\bmales?\b", "Men", text, flags=re.IGNORECASE)
    return text


def relabel(diffs, suffix):
    """Rename the non-reference group so two models can share one panel.

    Group names are mapped to Women/Men. The numeric prefix in the suffix
    forces the plotting order (primary before ablated) and is stripped from
    the legend afterwards.
    """
    return [
        {
            f"{group_labels.get(str(g).lower(), g)} ({suffix})": m
            for g, m in rep.items()
        }
        for rep in diffs
    ]


combined_diffs = [
    {**a, **b}
    for a, b in zip(
        relabel(diffs_primary, "1 with sex"),
        relabel(diffs_ablated, "2 sex removed"),
    )
]

captured = {}
real_show = plt.show
plt.show = lambda *a, **k: captured.setdefault("fig", plt.gcf())

try:
    eqb.eq_group_metrics_plot(
        group_metrics=combined_diffs,
        metric_cols=wanted_metrics,
        name="sex",
        categories="all",
        figsize=(8, 8),
        plot_type="violinplot",
        color_by_group=True,
        cmap="tab10",
        max_cols=2,
        show_grid=False,
        strict_layout=True,
        disparities=True,
        show_pass_fail=False,
    )
finally:
    plt.show = real_show

fig = captured.get("fig", plt.gcf())
fig.canvas.draw()  # populate tick labels before editing them

# Retitle metric panels, add (a)-(d) labels, and fix any female/male text
panel_letters = iter("abcd")
legends = list(fig.legends)

for ax in fig.axes:
    leg = ax.get_legend()
    if leg is not None:
        legends.append(leg)

    low = ax.get_title().lower()
    for key, new in title_map.items():
        if key in low:
            ax.set_title(new)
            ax.text(
                -0.12,
                1.05,
                f"({next(panel_letters)})",
                transform=ax.transAxes,
                fontsize=12,
                fontweight="bold",
                va="bottom",
                ha="left",
            )
            break

    for axis in (ax.xaxis, ax.yaxis):
        labels = [t.get_text() for t in axis.get_ticklabels()]
        if any(re.search(r"\b(fe)?males?\b", s, re.IGNORECASE) for s in labels):
            axis.set_ticks(axis.get_ticklocs())
            axis.set_ticklabels([to_people(s) for s in labels])

    ax.set_xlabel(to_people(ax.get_xlabel()))
    ax.set_ylabel(to_people(ax.get_ylabel()))

for leg in legends:
    for txt in leg.get_texts():
        s = txt.get_text()
        s = s.replace("1 with sex", "with sex").replace("2 sex removed", "sex removed")
        txt.set_text(to_people(s))

# Save in all four formats, creating each directory as needed
out = {
    "png": dict(dpi=300),
    "svg": dict(),
    "tiff": dict(dpi=600, pil_kwargs={"compression": "tiff_lzw"}),
    "pdf": dict(),
}

for ext, kw in out.items():
    p = Path(f"../images/{ext}_images/fig_6_disparity_violins.{ext}")
    p.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(p, bbox_inches="tight", **kw)

In [ ]:
def ci_table(sig, label):
    """Extract difference, interval and significance for each metric."""
    rows = []
    for metric, res in sig["Female"].items():
        lo, hi = res.confidence_interval or (None, None)
        rows.append({
            "metric": metric,
            f"{label}_diff": res.statistic,
            f"{label}_lo": lo,
            f"{label}_hi": hi,
            f"{label}_sig": res.is_significant,
        })
    return pd.DataFrame(rows).set_index("metric")


out = ci_table(sig_primary, "primary").join(ci_table(sig_ablated, "ablated"))
print(out.round(3).to_string())

In [ ]:
assert len(diffs_primary) == len(diffs_ablated)

def gap(reps):
    return np.array([r["Female"]["ROC_AUC_diff"] for r in reps], dtype=float)

g_pri, g_abl = gap(diffs_primary), gap(diffs_ablated)
did = g_abl - g_pri

lo, hi = np.percentile(did, [2.5, 97.5])
p = 2 * min((did <= 0).mean(), (did >= 0).mean())

print(f"n replicates     : {len(did)}")
print(f"Gap with sex     : {g_pri.mean():+.3f}")
print(f"Gap without sex  : {g_abl.mean():+.3f}")
print(f"Change in gap    : {did.mean():+.3f} ({lo:+.3f} to {hi:+.3f})")
print(f"p (two-sided)    : {min(p, 1.0):.4f}")

In [ ]:
np.corrcoef(g_pri, g_abl)[0, 1]

In [ ]:
"""
Stability of the group-wise AUC gap difference-in-differences across
repeated stratified splits.

Refits both models on each split rather than resampling one fixed test set,
so this answers a different question from the bootstrap: does the finding
survive a different partition of the cohort, not just a different resample
of the same held-out rows.

ROC AUC is threshold-independent, so no threshold selection is needed.
Hyperparameters are held at the tuned values; re-tuning inside each split
would answer a third question and cost far more.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score


def auc_gap(y, p, fem):
    """Female minus male AUC. NaN if either group lacks both classes."""
    male = ~fem
    if len(np.unique(y[fem])) < 2 or len(np.unique(y[male])) < 2:
        return np.nan
    return roc_auc_score(y[fem], p[fem]) - roc_auc_score(y[male], p[male])


def did_stability(X, y, sex_col, fit_predict, n_splits=20,
                  test_size=0.2, seed=222):
    """
    fit_predict(X_tr, y_tr, X_te, drop_sex) -> predicted probabilities on X_te.
    Supplied by the caller so this stays agnostic to the model API.

    Stratifies on sex x outcome so every split carries a comparable number of
    female events. Plain outcome stratification lets that count drift, which
    is the main thing that would make the estimate look unstable for reasons
    that have nothing to do with the finding.
    """
    X = X.reset_index(drop=True)
    y = np.asarray(y).astype(int)
    fem_all = (X[sex_col] == 0).to_numpy()

    strata = np.char.add(fem_all.astype(str), y.astype(str))
    splitter = StratifiedShuffleSplit(n_splits=n_splits, test_size=test_size,
                                      random_state=seed)

    rows = []
    for i, (tr, te) in enumerate(splitter.split(X, strata)):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y[tr], y[te]
        fem_te = fem_all[te]

        p_pri = fit_predict(X_tr, y_tr, X_te, drop_sex=False)
        p_abl = fit_predict(X_tr, y_tr, X_te, drop_sex=True)

        g_pri = auc_gap(y_te, np.asarray(p_pri), fem_te)
        g_abl = auc_gap(y_te, np.asarray(p_abl), fem_te)

        rows.append({
            "split": i,
            "n_female_events": int(y_te[fem_te].sum()),
            "n_male_events": int(y_te[~fem_te].sum()),
            "auc_pri": roc_auc_score(y_te, p_pri),
            "auc_abl": roc_auc_score(y_te, p_abl),
            "gap_pri": g_pri,
            "gap_abl": g_abl,
            "did": g_abl - g_pri,
        })

    return pd.DataFrame(rows)


def summarize(df):
    d = df["did"].dropna()
    return {
        "n_splits": len(d),
        "n_dropped": int(df["did"].isna().sum()),
        "did_median": float(d.median()),
        "did_mean": float(d.mean()),
        "did_iqr": (float(d.quantile(.25)), float(d.quantile(.75))),
        "did_range": (float(d.min()), float(d.max())),
        "frac_positive": float((d > 0).mean()),
    }